# NLP Robot Commander
## Commande intelligente et sûre d’un robot mobile

**Projet HESTIM — démonstration reproductible**  
Intentions : `navigate` · `stop` · `report`  
NLP : DistilBERT multilingue + règles de sûreté + LLM local Ollama + mémoire SQLite

> Notebook mis à jour le 26 octobre 2026. Les résultats chiffrés proviennent des artefacts versionnés et les essais Nav2 de la simulation réelle.

### Comment l’utiliser dans VS Code

1. Ouvrir le dépôt avec `code .` depuis Ubuntu 22.04 dans WSL.
2. Cliquer sur **Select Kernel** et choisir l’interpréteur `.venv`.
3. Cliquer sur **Run All** pour recalculer les tableaux et refaire les démonstrations locales.
4. Pour présenter sans recalculer, parcourir simplement les cellules : les principaux résultats sont déjà enregistrés.

Installation du noyau si nécessaire :

```bash
bash scripts/wsl_setup.sh
```

> Le runtime de référence est WSL2 + Ubuntu 22.04. Gazebo/Nav2 se lance séparément avec `bash scripts/wsl_run_simulation.sh`. Une seule instance de simulation doit tourner.

## 1. Problème et objectifs

Le système transforme une commande française en une décision robot structurée et vérifiée.

| Intention | Exemple | Action robot |
|---|---|---|
| `navigate` | « Va dans la cuisine » | But Nav2 `NavigateToPose` vers un waypoint autorisé |
| `stop` | « Arrête-toi immédiatement » | Annulation suivie du but Nav2 |
| `report` | « T’es où ? » | Lecture de la pose TF `map → base_link` |

Contraintes principales : refuser les commandes ambiguës, niées, séquentielles, hors domaine ou visant une destination inconnue.

## 2. Architecture du pipeline

```text
Texte / micro
     │
     ▼
Normalisation + règles de sûreté prioritaires
     │
     ├── commande explicite ───────────────► décision déterministe
     │
     └── formulation libre ─► Ollama ─────► JSON contraint
                              │
                              └─ indisponible ─► DistilBERT (secours configuré)
                                                   │
                                                   ▼
                    validation finale + liste blanche des waypoints
                                                   │
                      ┌────────────────────────────┼──────────────────────────┐
                      ▼                            ▼                          ▼
               NavigateToPose              annulation Nav2             rapport TF
```

Le LLM n’a aucun accès direct à ROS 2. Sa proposition est toujours contrôlée avant une éventuelle action.

In [1]:
from pathlib import Path
from dataclasses import asdict
import json
import subprocess
import sys

import pandas as pd

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'README.md').exists() and (candidate / 'src').exists():
            return candidate
    raise FileNotFoundError('Ouvrez le notebook depuis le dépôt NLP Robot Commander.')

ROOT = find_project_root(Path.cwd().resolve())
PACKAGE_ROOT = ROOT / 'src' / 'nlp_robot_commander'
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT))

print(f'Projet détecté : {ROOT.name}')
print('Package Python ajouté : src/nlp_robot_commander')

Projet détecté : NLP
Package Python ajouté : src/nlp_robot_commander


## 3. Dataset audité

Le corpus est équilibré entre les trois intentions et le test indépendant est séparé de l’entraînement.

| Ensemble | Total | navigate | stop | report |
|---|---:|---:|---:|---:|
| Entraînement + validation | 162 | 54 | 54 | 54 |
| Train | 126 | 42 | 42 | 42 |
| Validation | 36 | 12 | 12 | 12 |
| Test indépendant | 45 | 15 | 15 | 15 |

L’audit ne détecte aucun doublon exact ni quasi-doublon entre les splits au seuil de similarité `0,94`.

In [2]:
audit = json.loads((ROOT / 'artifacts/dataset_audit.json').read_text(encoding='utf-8'))
dataset = pd.read_csv(ROOT / 'dataset/dataset.csv')
test_dataset = pd.read_csv(ROOT / 'dataset/test_commands.csv')

print(f"Audit valide : {audit['ok']}")
print(f"Corpus entraînement/validation : {audit['training_rows']} phrases")
print(f"Test indépendant : {audit['test_rows']} phrases")
print(
    f"Doublons exacts : {len(audit['exact_duplicates'])} | "
    f"Quasi-doublons : {len(audit['near_duplicates'])}"
)
print('\nRépartition des intentions :')
print(dataset['intent'].value_counts().sort_index().to_string())

Audit valide : True
Corpus entraînement/validation : 162 phrases
Test indépendant : 45 phrases
Doublons exacts : 0 | Quasi-doublons : 0

Répartition des intentions :
intent
navigate    74
report      58
stop        60


## 4. Entraînement DistilBERT

- Modèle de base : `distilbert-base-multilingual-cased`
- Classes : `navigate`, `stop`, `report`
- Époques : **12**
- Taille de lot : **8**
- Learning rate : **2 × 10⁻⁵**
- Graine : **42**
- Seuil d’acceptation : **0,65**

Le checkpoint est local et n’est pas versionné sur GitHub à cause de sa taille. Les scripts permettent de le réentraîner.

In [3]:
validation = json.loads(
    (ROOT / 'artifacts/training_3intents/validation_metrics.json').read_text(encoding='utf-8')
)
independent_test = json.loads(
    (ROOT / 'artifacts/test_campaign_3intents/metrics.json').read_text(encoding='utf-8')
)
safety = json.loads(
    (ROOT / 'artifacts/safety_campaign_transformers/metrics.json').read_text(encoding='utf-8')
)

results = pd.DataFrame([
    {
        'Jeu': 'Validation DistilBERT',
        'Taille': validation['validation_size'],
        'Accuracy': f"{validation['accuracy']:.2%}",
        'F1 macro': f"{validation['classification_report']['macro avg']['f1-score']:.2%}",
    },
    {
        'Jeu': 'Test indépendant',
        'Taille': independent_test['sample_count'],
        'Accuracy': f"{independent_test['accuracy']:.2%}",
        'F1 macro': f"{independent_test['classification_report']['macro avg']['f1-score']:.2%}",
    },
    {
        'Jeu': 'Campagne de sûreté',
        'Taille': safety['sample_count'],
        'Accuracy': f"{safety['pass_rate']:.2%}",
        'F1 macro': '—',
    },
])
print(results.to_string(index=False))

                  Jeu  Taille Accuracy F1 macro
Validation DistilBERT      36   97.22%   97.22%
     Test indépendant      45   97.78%   97.78%
   Campagne de sûreté      15  100.00%        —


## 5. Matrices de confusion mesurées

### Validation — 36 phrases

![Matrice de confusion validation](artifacts/training_3intents/confusion_matrix.png)

### Test indépendant — 45 phrases

![Matrice de confusion test](artifacts/test_campaign_3intents/confusion_matrix.png)

Sur la validation, une phrase `navigate` a été prédite `stop`. Sur le test indépendant, **44 intentions sur 45** sont correctement classées. La formulation « Localisation actuelle s’il te plaît » est prédite `stop` avec une confiance de `0,49`, puis refusée par le seuil de sécurité `0,65`.

## 6. Démonstration locale du pipeline sûr

Cette cellule utilise le classifieur lexical léger pour rester rapide et reproductible. Le processeur, l’extracteur d’entités et les règles sont exactement ceux du package ROS 2, y compris les formulations familières ajoutées après les essais utilisateur.

In [4]:
from nlp_robot_commander.command_processor import CommandProcessor
from nlp_robot_commander.entity_extractor import EntityExtractor
from nlp_robot_commander.intent_classifier import RuleBasedIntentClassifier

extractor = EntityExtractor.from_yaml(
    ROOT / 'src/nlp_robot_commander/config/waypoints.yaml'
)
processor = CommandProcessor(RuleBasedIntentClassifier(), extractor)
commands = [
    'Va cuisine',
    "t'est ou ?",
    'ou est tu?',
    'Arrête-toi immédiatement',
    'Va dans la chambre',
    'Prends la boîte rouge',
]
rows = []
for command in commands:
    decision = processor.process(command)
    rows.append({
        'commande': command,
        'acceptée': decision.accepted,
        'intention': decision.intent,
        'destination': decision.destination or '—',
        'source': decision.source,
        'erreur': decision.error_code or '—',
    })
print(pd.DataFrame(rows).to_string(index=False))

                commande  acceptée intention destination              source              erreur
              Va cuisine      True  navigate     cuisine deterministic_rules                   —
              t'est ou ?      True    report           — deterministic_rules                   —
              ou est tu?      True    report           — deterministic_rules                   —
Arrête-toi immédiatement      True      stop           —        safety_rules                   —
      Va dans la chambre     False  navigate           —          classifier UNKNOWN_DESTINATION
   Prends la boîte rouge     False   unknown           —          classifier  UNSUPPORTED_ACTION


### Interprétation

- Les commandes explicites sûres sont traitées sans dépendre d’un modèle génératif.
- `t'est ou ?` et `ou est tu?` sont reconnus comme `report`, même avec ces formulations familières.
- Une destination inconnue est refusée.
- L’action `pick` est hors du périmètre actuel et ne peut pas atteindre ROS 2.
- Un arrêt explicite a la priorité et passe par les règles de sûreté.

## 7. Mémoire locale SQLite

La mémoire ne réentraîne pas le LLM. Elle enregistre au maximum 200 commandes acceptées et permet de résoudre une référence explicite comme « refais la dernière commande ».

La répétition implicite d’un ancien `stop` est volontairement interdite : un arrêt doit toujours être redemandé explicitement.

In [5]:
from nlp_robot_commander.command_memory import CommandMemory

memory_path = ROOT / 'artifacts/runtime/notebook_presentation_memory.sqlite3'
memory_path.parent.mkdir(parents=True, exist_ok=True)
memory = CommandMemory(memory_path)
memory.clear()
try:
    memory_processor = CommandProcessor(
        RuleBasedIntentClassifier(), extractor, memory=memory
    )
    first = memory_processor.process('Va cuisine')
    memory_processor.remember('Va cuisine', first)
    repeated = memory_processor.process('Refais la dernière commande')
    print(
        f'1. {first.intent} -> {first.destination} | '
        f'source={first.source} | memory_used={first.memory_used}'
    )
    print(
        f'2. {repeated.intent} -> {repeated.destination} | '
        f'source={repeated.source} | memory_used={repeated.memory_used}'
    )
finally:
    memory.close()
    memory_path.unlink(missing_ok=True)

1. navigate -> cuisine | source=deterministic_rules | memory_used=False
2. navigate -> cuisine | source=memory_rules | memory_used=True


## 8. LLM local Ollama — démonstration optionnelle

Le modèle configuré est `llama3.2:1b`, servi localement par Ollama sur `127.0.0.1:11434`. Aucune clé API ni donnée cloud n’est nécessaire.

Le LLM sert à interpréter des formulations libres. Sa sortie suit un schéma JSON fermé : intention, confiance, destination autorisée et utilisation éventuelle de la mémoire. Le processeur peut encore refuser cette sortie.

Passez `RUN_LLM` à `True` uniquement si `ollama list` affiche `llama3.2:1b`.

In [6]:
from nlp_robot_commander.llm_interpreter import OllamaLlmInterpreter

RUN_LLM = False
if RUN_LLM:
    llm = OllamaLlmInterpreter(
        tuple(extractor.waypoints), model='llama3.2:1b', timeout_sec=60
    )
    llm_processor = CommandProcessor(
        RuleBasedIntentClassifier(), extractor, llm_interpreter=llm
    )
    llm_result = llm_processor.process(
        "Pourrais-tu m'indiquer précisément où tu te trouves actuellement ?"
    )
    print(json.dumps(asdict(llm_result), ensure_ascii=False, indent=2))
else:
    print(
        "Démo LLM désactivée pour une présentation reproductible. "
        "Définissez RUN_LLM = True pour l'activer."
    )

Démo LLM désactivée pour une présentation reproductible. Définissez RUN_LLM = True pour l'activer.


## 9. Tests automatisés

État vérifié lors de la mise à jour du notebook :

| Vérification | Résultat |
|---|---:|
| Tests unitaires et d’intégration locale | **63 réussis** |
| Compilation Python | **réussie** |
| Vérification du diff Git | **réussie** |

La cellule suivante relance les tests. Elle ne modifie ni le modèle ni les données.

In [7]:
# Cette cellule prend quelques secondes. Décommentez pour relancer les 63 tests.
# completed = subprocess.run(
#     [sys.executable, '-m', 'pytest', '-q', '--basetemp', '.pytest-notebook'],
#     cwd=ROOT,
#     text=True,
#     capture_output=True,
# )
# print(completed.stdout or completed.stderr)
# assert completed.returncode == 0

## 10. Intégration ROS 2 / Nav2

Sous WSL2 avec Ubuntu 22.04 et ROS 2 Humble :

```bash
bash scripts/wsl_build.sh
bash scripts/wsl_run_simulation.sh
```

Terminal de commandes :

```bash
source scripts/wsl_env.sh
ros2 run nlp_robot_commander terminal_client
```

Événements observables :

```bash
source scripts/wsl_env.sh
ros2 topic echo /nlp_commander/status
```

Le monde par défaut représente un appartement avec salon, cuisine, chambre, bureau et couloir central. Les zones d’arrivée ont été recentrées derrière les portes après un test réel terminé avec le statut Nav2 `SUCCEEDED`.

![Plan de l’appartement et waypoints sûrs](docs/apartment_layout.svg)

| Destination | x | y | yaw |
|---|---:|---:|---:|
| cuisine | 2,50 | 1,40 | 1,57 |
| salon | -2,50 | 1,40 | 1,57 |
| bureau | 2,50 | -1,40 | -1,57 |
| chargeur | -4,00 | 0,00 | 0,00 |

## 11. Limites et lecture honnête des résultats

- Le score de **97,78 %** concerne 45 phrases de test indépendantes, pas toutes les formulations possibles du français.
- Les waypoints fournis sont calibrés pour l’appartement du projet et doivent être recalibrés pour une autre carte.
- Les résultats NLP ne prouvent pas à eux seuls le succès physique de Nav2.
- La navigation vers la cuisine a été validée dans Gazebo avec `SUCCEEDED`, mais ce résultat ne garantit pas toutes les cartes ni un robot réel.
- L’entrée audio reste optionnelle et nécessite une campagne dédiée au bruit et aux accents.
- La mémoire SQLite fournit du contexte ; elle ne réalise pas d’apprentissage en ligne des poids du LLM.
- Une réponse du LLM reste filtrée par les règles déterministes avant le routage robot.

## 12. Conclusion

Le prototype fournit une chaîne complète et testable :

1. compréhension de trois intentions en français ;
2. extraction de quatre destinations autorisées ;
3. DistilBERT évalué sur un test indépendant ;
4. LLM local optionnel avec sortie structurée ;
5. mémoire locale explicite et contrôlée ;
6. barrières de sûreté avant ROS 2/Nav2 ;
7. package ROS 2 validé dans l’appartement Gazebo avec des waypoints sûrs.

### Chiffres à retenir

**97,22 %** en validation · **97,78 %** sur 45 tests indépendants · **100 %** sur 15 tests de sûreté · **63 tests automatisés réussis** · **Nav2 SUCCEEDED** vers la cuisine

## Déroulé conseillé pour la soutenance

- **1 min** — problème, intentions et contraintes de sûreté ;
- **2 min** — architecture DistilBERT + LLM + mémoire + ROS 2 ;
- **2 min** — dataset, métriques et matrices de confusion ;
- **2 min** — exécuter les cellules de démonstration des commandes et de la mémoire ;
- **1 min** — montrer la vidéo ou la simulation Gazebo/Nav2 ;
- **1 min** — limites, conclusion et questions.

Documents complémentaires : `docs/rapport_final.md`, `docs/checklist_demo.md` et `docs/HOW_TO_FIRST_RUN_ON_VSCODE.md`.

Support PowerPoint : [`output/pptx/NLP_Robot_Commander_Soutenance.pptx`](output/pptx/NLP_Robot_Commander_Soutenance.pptx).